# **Подготовка к выполнению ДЗ**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from tqdm import tqdm
import matplotlib.pyplot as plt

from torch.utils.tensorboard import SummaryWriter
import datetime
import requests
import zipfile
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [ ]:
data_url = "https://disk.yandex.ru/d/uW1XErC4YM3Wpg"

def download_from_yandex_disk(public_url, output_path):
    api_url = f"https://cloud-api.yandex.net/v1/disk/public/resources?public_key={public_url}"

    response = requests.get(api_url)
    data = response.json()
    download_url = data['file']

    response = requests.get(download_url, stream=True)
    if response.status_code != 200:
        print(f"Ошибка: {response.status_code}")
        print(f"Ответ: {response.text}")
        return False

    total_size = int(response.headers.get('content-length', 0))

    with open(output_path, 'wb') as f:
        downloaded = 0
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            downloaded += len(chunk)
            if total_size > 0:
                percent = (downloaded / total_size) * 100
                print(f"\rПрогресс: {percent:.1f}% ({downloaded}/{total_size} bytes)", end='')

    print(f"\nФайл сохранён: {output_path}")
    return True

download_from_yandex_disk(data_url, '/content/data.zip')
print(f"\nРазмер файла: {os.path.getsize('/content/data.zip') / (1024*1024):.2f} MB")


with zipfile.ZipFile('/content/data.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/ai-vs-human-generated-dataset-hw/')


Прогресс: 100.0% (1149185420/1149185420 bytes)
Файл сохранён: /content/data.zip

Размер файла: 1095.95 MB


In [ ]:
class ImageDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.root_dir = Path(root_dir)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = self.root_dir / self.data.iloc[idx]['file_name']
        image = Image.open(img_path).convert('RGB')
        label = self.data.iloc[idx]['label']

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

parent_dir = "/content/ai-vs-human-generated-dataset-hw/ai-vs-human-generated-dataset-hw"

train_dataset = ImageDataset(
    csv_file = parent_dir + '/Train_1/train.csv',
    root_dir = parent_dir + '/Train_1',
    transform=train_transform
)

test_dataset = ImageDataset(
    csv_file = parent_dir + '/Test_1/test.csv',
    root_dir = parent_dir + '/Test_1',
    transform=test_transform
)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

print(f'Train dataset size: {len(train_dataset)}')
print(f'Test dataset size: {len(test_dataset)}')

Train dataset size: 9993
Test dataset size: 3997


In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)
model = model.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 148MB/s]


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

# **Пункты 1-2**

In [ ]:
!pip install tensorboard -q

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in tqdm(dataloader, desc='Training'):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    # добавьте подсчет epoch_loss, epoch_acc, epoch_f1
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')

    return epoch_loss, epoch_acc, epoch_f1

In [ ]:
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc='Validation'):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # добавьте подсчет epoch_loss, epoch_acc, epoch_f1, epoch_precision, epoch_recall
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
    epoch_precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    epoch_recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)


    return epoch_loss, epoch_acc, epoch_f1, epoch_precision, epoch_recall

In [ ]:
num_epochs = 10
train_losses = []
train_accs = []
train_f1s = []

# Добавьте логгирование метрик для tensorboard
log_dir = f"logs/experiment_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
writer = SummaryWriter(log_dir=log_dir)

training_params = {
    'model': 'resnet18',
    'pretrained': True,
    'batch_size': batch_size,
    'num_epochs': num_epochs,
    'learning_rate': 0.001,
    'optimizer': 'Adam',
    'scheduler': 'StepLR',
    'step_size': 5,
    'gamma': 0.1,
    'train_dataset_size': len(train_dataset),
    'test_dataset_size': len(test_dataset),
    'device': str(device)
}
for key, value in training_params.items():
    writer.add_text('Training Parameters', f'{key}: {value}', 0)

for epoch in range(num_epochs):
    print(f'\nEpoch {epoch+1}/{num_epochs}')
    print('-' * 50)

    train_loss, train_acc, train_f1 = train_epoch(model, train_loader, criterion, optimizer, device)
    scheduler.step()

    train_losses.append(train_loss)
    train_accs.append(train_acc)
    train_f1s.append(train_f1)

    writer.add_scalar('Loss/train', train_loss, epoch)
    writer.add_scalar('Accuracy/train', train_acc, epoch)
    writer.add_scalar('F1/train', train_f1, epoch)
    writer.add_scalar('Learning_rate', optimizer.param_groups[0]['lr'], epoch)

    print(f'Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}')

writer.close()
print('\nTraining completed!')


Epoch 1/10
--------------------------------------------------


Training: 100%|██████████| 313/313 [01:56<00:00,  2.70it/s]


Train Loss: 0.3520, Acc: 0.8538, F1: 0.8538

Epoch 2/10
--------------------------------------------------


Training: 100%|██████████| 313/313 [01:24<00:00,  3.71it/s]


Train Loss: 0.2660, Acc: 0.8908, F1: 0.8908

Epoch 3/10
--------------------------------------------------


...

Train Loss: 0.0906, Acc: 0.9641, F1: 0.9641

Epoch 9/10
--------------------------------------------------


Training: 100%|██████████| 313/313 [01:27<00:00,  3.60it/s]


Train Loss: 0.0772, Acc: 0.9705, F1: 0.9705

Epoch 10/10
--------------------------------------------------


Training: 100%|██████████| 313/313 [01:26<00:00,  3.60it/s]

Train Loss: 0.0707, Acc: 0.9737, F1: 0.9737

Training completed!


почему-то csv-шник внутри папки Test_1, который по идее должен описывать пути к тестовым фотографиям, содержит пути к файлам из папки train_data.

Поэтому оценку модели я проводил на фотографиях которые лежат по путям, получаемым заменой префикса "train_data" на "test_data"

![](reports/path_trouble_report.png)

In [ ]:
test_dataset.data['file_name'] = test_dataset.data['file_name'].str.replace('train_data', 'test_data')

In [ ]:
# Добавьте логику оценки модели на тестовом датасете, как метрику в tensorboard
# pip install tensorboard
# команда для поднятия
# tensorboard --logdir=my_logs

test_loss, test_acc, test_f1, test_precision, test_recall = validate(model, test_loader, criterion, device)

writer_test = SummaryWriter(log_dir=f"{log_dir}/test_evaluation")
writer_test.add_scalar('Test/Loss', test_loss, 0)
writer_test.add_scalar('Test/Accuracy', test_acc, 0)
writer_test.add_scalar('Test/F1', test_f1, 0)
writer_test.add_scalar('Test/Precision', test_precision, 0)
writer_test.add_scalar('Test/Recall', test_recall, 0)

writer_test.close()

Validation: 100%|██████████| 125/125 [00:34<00:00,  3.65it/s]


In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

# **Пункт 3**

Перехожу из гугл-колаба на свое железо, потому что докер не хочет запускаться.

In [ ]:
model_path = 'models/model_v1.pth'
os.makedirs('models', exist_ok=True)

test_metrics = {
    'loss': test_loss,
    'accuracy': test_acc,
    'f1': test_f1,
    'precision': test_precision,
    'recall': test_recall
}

torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_losses': train_losses,
    'train_accs': train_accs,
    'train_f1s': train_f1s,
    'test_metrics': test_metrics,
    'training_params': training_params
}, model_path)

In [ ]:
from google.colab import files
files.download('models/model_v1.pth')

!zip -r logs.zip logs/
files.download('logs.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  adding: logs/ (stored 0%)
  adding: logs/experiment_20260512_163343/ (stored 0%)
  adding: logs/experiment_20260512_163343/events.out.tfevents.1778603623.a4ae2277b1f3.628.0 (deflated 61%)
  adding: logs/experiment_20260512_163343/test_evaluation/ (stored 0%)
  adding: logs/experiment_20260512_163343/test_evaluation/events.out.tfevents.1778605945.a4ae2277b1f3.628.3 (deflated 21%)
  adding: logs/experiment_20260512_163343/test_evaluation/.ipynb_checkpoints/ (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Результат загрузки модели в S3 (MinIO)**

есть два скрипта:
- `s3_scripts\upload_to_s3.py` - для загрузки модели в облако
- `s3_scripts\download_from_s3.py` - для выгрузки модели из облака

Так как обучения происходит в колабе (иначе мой компик будет учить эпоху 4 часа) и там есть проблема с поднятием контейнера, я сделал скрипты которые бы автоматизирвоали работу с s3 и продемонстрировал их работу (см. скриншоты ниже).

Но загрузка и выгрузка моделей внутри пайплайна dvc из пункта 4 будет эмулироваться с помощью "локальной загрузки" моделей, по причинам описанным выше.

Елси фотка ниже не грузится, то ее можно найти в `reports/s3_report.png`

![Модель в S3](reports/s3_report.png)

# **Пункт 4**

Пайплайн DVC следующий:
- выгрузка модели с s3 (файл `dvc_pipeline/download_model.py`).  Копирует модель в `models/model_v1_pretrained.pth`
- Дообучение. (файл `dvc_pipeline/finetune.py`)
- Оценка модели. (файл `dvc_pipeline/test_model.py`). Тестирует дообученную модель и логирует процесс.
- Выгрузка получившийся модели. (файл `dvc_pipeline/upload_model.py`)

По сути все эти файлы содеражт перегруппированный код из этого ноутбука.

Как и говорилось выше, файлы `download_model.py` и `upload_model.py` лишь эмулируют работу с s3. Если бы была возможность поднимать контейнер, то процесс взаимодействия с s3 внутри dvc пайплайна выглядел бы как в предыдущем пункте.

In [34]:
# Добавьте логику дообучения
# Добавьте PVC пайплайн
!pip install dvc -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.1/470.1 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.2/451.2 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.2/214.2 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.7/

In [44]:
%cd /content
!dvc init --no-scm

/content
Initialized DVC repository.

+---------------------------------------------------------------------+
|                                                                     |
|        DVC has enabled anonymous aggregate usage analytics.         |
|     Read the analytics documentation (and how to opt-out) here:     |
|             <https://dvc.org/doc/user-guide/analytics>              |
|                                                                     |
+---------------------------------------------------------------------+

What's next?
------------
- Check out the documentation: <https://dvc.org/doc>
- Get help and share ideas: <https://dvc.org/chat>
- Star us on GitHub: <https://github.com/treeverse/dvc>


In [46]:
%cd /content
!dvc repro

/content
Stage 'download_from_s3' didn't change, skipping
Running stage 'finetune':
> python dvc_pipeline/finetune.py
DVC piplene.	Step 2: Дообучение модели на Train_2
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Train dataset size: 3997
Using device: cuda
Загружена предобученная модель: models/model_v1_pretrained.pth
TensorBoard лог: logs/finetune_20260512_194200
Начало дообучения

Epoch 1/5
----------------------------------------
Fine-tuning:   0% 0/125 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: 

In [47]:
%cd /content
!dvc dag

/content
    +------------------+         
    | download_from_s3 |         
    +------------------+         
              *                  
              *                  
              *                  
        +----------+             
        | finetune |             
        +----------+             
         *         *             
       **           **           
      *               *          
+------+        +--------------+ 
| test |        | upload_to_s3 | 
+------+        +--------------+ 


# **Пункты 5-6**

Посомтрим на логи обучения и дообучения

In [ ]:
%reload_ext tensorboard
%tensorboard --logdir logs --port 6008

![Модель в S3](reports/logs_report.png)

Как мы видим по логам, дообученная модель превосходит базовую по всем тестовым метрикам!

| Метрика | Model_1 | Model_2 |
|---------|---------|---------|
| **Accuracy/train** | 0.965 | **0.967** | 
| **F1/train** | 0.954 | **0.969** | 
| **Loss/train** | **0.070** | 0.074 | 
| **Accuracy/test** | 0.965 | **0.985** |
| **F1/test** | 0.965 | **0.985** | 
| **Loss/test** | 0.0955 | **0.0581** | 
| **Precision/test** | 0.966 | **0.985** | 
| **Recall/test** | 0.965 | **0.985** | 

**Пайплайн успешно отработал.** Все этапы MLOps пайплайна реализованы:
1. Обучение и валидация моделей
2. Логирование метрик в TensorBoard
3. Сохранение артефактов в S3 (MinIO)
4. Воспроизводимый DVC пайплайн